In [ ]:
#import libraries
import pandas as pd
import numpy as np
import re
import string


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report


In [ ]:
# Load Dataset
train_df = pd.read_csv("twitter_training.csv", header=None)
val_df = pd.read_csv("twitter_validation.csv", header=None)

train_df.columns = ['ID', 'Entity', 'Sentiment', 'Text']
val_df.columns = ['ID', 'Entity', 'Sentiment', 'Text']

df = pd.concat([train_df, val_df], ignore_index=True)
df = df[df['Sentiment'].isin(['Positive', 'Negative', 'Neutral'])]
df = df[df['Text'].apply(lambda x: isinstance(x, str))].copy()

In [ ]:
# Preprocess Text
def preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    text = re.sub(r'\@w+|\#', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['Clean_Text'] = df['Text'].apply(preprocess)


In [ ]:

# Vectorization
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=10000,
    ngram_range=(1, 2),
    max_df=0.95,
    min_df=2
)
X = vectorizer.fit_transform(df['Clean_Text'])

label_map = {'Positive': 1, 'Negative': -1, 'Neutral': 0}
y = df['Sentiment'].map(label_map)


In [ ]:

#Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Logistic Regression
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],
    'solver': ['saga', 'liblinear']
}


grid_search = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight='balanced'),
    param_grid,
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)


Fitting 5 folds for each of 10 candidates, totalling 50 fits


GridSearchCV(cv=5,
             estimator=LogisticRegression(class_weight='balanced',
                                          max_iter=1000),
             n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l2'],
                         'solver': ['saga', 'liblinear']},
             verbose=1)

In [ ]:

# Train model
lr_model = grid_search.best_estimator_
lr_preds = lr_model.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_preds))
print(classification_report(y_test, lr_preds))


Logistic Regression Accuracy: 0.8475383373688459
              precision    recall  f1-score   support

          -1       0.89      0.85      0.87      4525
           0       0.79      0.85      0.82      3679
           1       0.86      0.84      0.85      4186

    accuracy                           0.85     12390
   macro avg       0.85      0.85      0.85     12390
weighted avg       0.85      0.85      0.85     12390



In [ ]:

# Testing with new sample text
new_text = ["This is an amazing product! Highly recommended.", "Worst purchase I made. Not worth it."]
new_text_cleaned = [preprocess(text) for text in new_text]  # Preprocess the text

new_text_transformed = vectorizer.transform(new_text_cleaned)

predictions = lr_model.predict(new_text_transformed)


reverse_label_map = {1: 'Positive', -1: 'Negative', 0: 'Neutral'}
predicted_labels = [reverse_label_map[pred] for pred in predictions]

for text, label in zip(new_text, predicted_labels):
    print(f"Text: {text}\nPredicted Sentiment: {label}\n")


Text: This is an amazing product! Highly recommended.
Predicted Sentiment: Positive

Text: Worst purchase I made. Not worth it.
Predicted Sentiment: Negative



In [ ]:
#save model
import joblib

joblib.dump(lr_model, "logistic_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")


['tfidf_vectorizer.pkl']

In [ ]:
from google.colab import files
files.download('/content/logistic_model.pkl')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download('/content/tfidf_vectorizer.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>